## Шаг 0. Клонирование репозитория (Colab / Kaggle)

На Colab и Kaggle локальной копии репо нет — первой ячейкой клонируем его и добавляем `src/` в `sys.path`, чтобы `from gp1.env import ...` работал. Локально ячейка — no-op.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/DKazhekin/ITMO_Speech_Recognition_Course.git"
REPO_NAME = "ITMO_Speech_Recognition_Course"

_in_colab = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ
_in_kaggle = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if _in_colab or _in_kaggle:
    repo_dir = Path(REPO_NAME)
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    gp1_src = (repo_dir / "group-projects/gp1/src").resolve()
    if str(gp1_src) not in sys.path:
        sys.path.insert(0, str(gp1_src))
    print(f"Repo cloned, gp1 src on sys.path: {gp1_src}")
else:
    print("Local run — repo already checked out.")

# 01b. QuartzNet + WordAux — вспомогательный CTC-loss на уровне слов

## Идея

Помимо основного char-CTC, добавляем вспомогательный CTC-loss, который работает
на уровне слов (WordAuxCTCHead). Энкодер проецирует свои финальные фичи в
пространство closed word-vocabulary (43 слова + blank).

Это даёт дополнительный градиентный сигнал: модель обучается одновременно
предсказывать символы и слова — эффект схож с multi-task learning.

**Адаптация под Wave-1 Batch:** поля `word_targets` / `word_target_lengths` удалены из `Batch`.
Word-targets вычисляются **на лету в цикле** из `batch.transcriptions` через `digits_to_words` + `WordVocab.encode`.
`WordVocab` и `NUMBER_WORDS` — inline в этом ноутбуке (удалены из src/ в Wave 1).

In [ ]:
import sys
import os

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "..", "src"))

from gp1.env import setup_environment
paths, device = setup_environment()
print(f"Platform OK. Device: {device}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

from gp1.data.manifest import records_from_csv
from gp1.data.dataset import SpokenNumbersDataset
from gp1.data.collate import collate_fn
from gp1.data.audio_aug import AudioAugmenter
from gp1.data.spec_aug import SpecAugmenter
from gp1.features.melbanks import LogMelFilterBanks
from gp1.losses.ctc import CTCLoss
from gp1.models.quartznet import QuartzNet10x4
from gp1.train.optim import build_novograd
from gp1.train.schedulers import build_cosine_warmup
from gp1.train.checkpoint import save_best
from gp1.train.metrics import compute_cer, compute_per_speaker_cer
from gp1.decoding.greedy import greedy_decode
from gp1.text.vocab import CharVocab
from gp1.text.normalize import digits_to_words
from gp1.types import AugConfig

In [ ]:
# ---------------------------------------------------------------------------
# WordVocab + NUMBER_WORDS — скопировано из src/gp1/text/vocab_word.py (удалён в Wave 1)
# ---------------------------------------------------------------------------

# Exhaustive set of Russian number words for range 1000..999999.
NUMBER_WORDS: tuple[str, ...] = (
    "ноль",
    "один",
    "одна",
    "два",
    "две",
    "три",
    "четыре",
    "пять",
    "шесть",
    "семь",
    "восемь",
    "девять",
    "десять",
    "одиннадцать",
    "двенадцать",
    "тринадцать",
    "четырнадцать",
    "пятнадцать",
    "шестнадцать",
    "семнадцать",
    "восемнадцать",
    "девятнадцать",
    "двадцать",
    "тридцать",
    "сорок",
    "пятьдесят",
    "шестьдесят",
    "семьдесят",
    "восемьдесят",
    "девяносто",
    "сто",
    "двести",
    "триста",
    "четыреста",
    "пятьсот",
    "шестьсот",
    "семьсот",
    "восемьсот",
    "девятьсот",
    "тысяча",
    "тысячи",
    "тысяч",
)

_WORD_TO_ID: dict[str, int] = {word: idx + 1 for idx, word in enumerate(NUMBER_WORDS)}
_ID_TO_WORD: dict[int, str] = {v: k for k, v in _WORD_TO_ID.items()}


class WordVocab:
    """Closed word vocabulary for Russian number words. BLANK_ID = 0."""

    BLANK_ID: int = 0

    @property
    def size(self) -> int:
        return 1 + len(NUMBER_WORDS)

    def encode(self, text: str) -> list[int]:
        """Encode space-separated Russian number words to token ids."""
        if not text:
            return []
        result: list[int] = []
        for word in text.split():
            if word not in _WORD_TO_ID:
                raise ValueError(f"Word {word!r} is not in the closed vocabulary.")
            result.append(_WORD_TO_ID[word])
        return result

    def decode(self, ids: list[int]) -> str:
        """Decode token ids to space-separated word string (strips blank ids)."""
        if not ids:
            return ""
        words: list[str] = []
        for token_id in ids:
            if token_id == self.BLANK_ID:
                continue
            words.append(_ID_TO_WORD[token_id])
        return " ".join(words)

In [ ]:
# ---------------------------------------------------------------------------
# WordAuxCTCHead — скопировано из src/gp1/losses/word_aux.py (удалён в Wave 1)
# Адаптация: использует out.intermediate вместо отдельного поля Batch.
# ---------------------------------------------------------------------------

class WordAuxCTCHead(nn.Module):
    """Auxiliary CTC head for word-level targets applied to encoder features.

    Adapted for Wave-1 Batch: word targets are computed on-the-fly in the
    training loop from batch.transcriptions, not stored in Batch fields.

    Args:
        d_enc: Encoder feature dimension.
        word_vocab_size: Size of the word vocabulary (WordVocab.size).
        blank_id: CTC blank token id (default 0).
    """

    def __init__(self, d_enc: int, word_vocab_size: int, blank_id: int = 0) -> None:
        super().__init__()
        self.proj = nn.Linear(d_enc, word_vocab_size)
        self._ctc = CTCLoss(blank_id=blank_id)

    def forward(
        self,
        enc_features: torch.Tensor,        # [B, T, d_enc]
        input_lengths: torch.Tensor,       # [B]
        word_targets: torch.Tensor,        # [B, U_word]
        word_target_lengths: torch.Tensor, # [B]
    ) -> torch.Tensor:
        assert enc_features.dim() == 3, "enc_features must be [B, T, D]"
        logits = self.proj(enc_features)
        log_probs = F.log_softmax(logits.float(), dim=-1)
        return self._ctc(
            log_probs,
            targets=word_targets,
            input_lengths=input_lengths,
            target_lengths=word_target_lengths,
        )

In [ ]:
# --- Вспомогательная функция: кодирование word-targets для батча ---

def _encode_words(
    trans_list: list[str],
    wvocab: WordVocab,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Encode a list of digit-string transcriptions into word-level token tensors.

    Args:
        trans_list: List of digit strings (e.g. ["139473", "25000"]).
        wvocab: WordVocab instance.

    Returns:
        (tensor [B, U_max], lengths [B]) — padded int64 tensors.
    """
    encoded = [wvocab.encode(digits_to_words(t)) for t in trans_list]
    lengths = torch.tensor([len(e) for e in encoded], dtype=torch.int64)
    u_max = max(lengths.max().item(), 1)
    tensor = torch.zeros(len(encoded), u_max, dtype=torch.int64)
    for i, e in enumerate(encoded):
        tensor[i, : len(e)] = torch.tensor(e, dtype=torch.int64)
    return tensor, lengths

In [ ]:
# --- Manifests + vocab ---
all_records = records_from_csv(paths.train_csv, paths.train_root)
dev_records = records_from_csv(paths.dev_csv, paths.dev_root)
vocab = CharVocab()
word_vocab = WordVocab()
print(f"Train: {len(all_records)} records. Dev: {len(dev_records)} records.")
print(f"CharVocab size: {vocab.vocab_size}, WordVocab size: {word_vocab.size}")

In [ ]:
HP_FIXED = dict(samplerate=16000, n_mels=80, n_fft=512, hop_length=160, win_length=400)

HP = dict(
    lr=0.01,
    weight_decay=1e-3,
    dropout=0.1,
    d_model=256,
    subsample_factor=2,
    warmup_steps=3000,
    word_aux_weight=0.2,
    max_epochs=30,
    grad_accum=2,
    batch_size=32,
    speed_prob=0.5,
    gain_prob=0.7,
    noise_prob=0.0,
    freq_mask_param=15,
    time_mask_param=25,
)
print("HP:", HP)

In [ ]:
aug_cfg = AugConfig(speed_prob=HP["speed_prob"], gain_prob=HP["gain_prob"],
                    noise_prob=HP["noise_prob"], seed=42)
train_augmenter = AudioAugmenter(aug_cfg)

train_ds = SpokenNumbersDataset(records=all_records, vocab=vocab,
                                 target_samplerate=HP_FIXED["samplerate"],
                                 augmenter=train_augmenter)
dev_ds = SpokenNumbersDataset(records=dev_records, vocab=vocab,
                               target_samplerate=HP_FIXED["samplerate"], augmenter=None)

train_loader = DataLoader(train_ds, batch_size=HP["batch_size"], shuffle=True,
                           num_workers=2, collate_fn=collate_fn, pin_memory=True)
dev_loader = DataLoader(dev_ds, batch_size=HP["batch_size"], shuffle=False,
                         num_workers=2, collate_fn=collate_fn)
print(f"Train batches: {len(train_loader)}, Dev batches: {len(dev_loader)}")

## Развёрнутый тренировочный цикл — с Word-Aux CTC

**Best practice — добавлен `torch.nn.utils.clip_grad_norm_`** с `max_norm=1.0` перед каждым шагом оптимизатора.
Предотвращает взрывной рост градиентов при суммировании char-CTC и word-CTC.
Значение `max_norm=1.0` — стандартное для CTC-задач с NovoGrad.

In [ ]:
mel_extractor = LogMelFilterBanks(
    n_fft=HP_FIXED["n_fft"], samplerate=HP_FIXED["samplerate"],
    hop_length=HP_FIXED["hop_length"], win_length=HP_FIXED["win_length"],
    n_mels=HP_FIXED["n_mels"],
).to(device)
spec_aug = SpecAugmenter(freq_mask_param=HP["freq_mask_param"], num_freq_masks=2,
                          time_mask_param=HP["time_mask_param"], num_time_masks=5, seed=42)

model = QuartzNet10x4(
    vocab_size=vocab.vocab_size, d_model=HP["d_model"],
    dropout=HP["dropout"], subsample_factor=HP["subsample_factor"],
).to(device)

# WordAuxCTCHead работает на финальных фичах энкодера.
# d_enc = 512 (последний блок QuartzNet10x4 возвращает log_probs из 512-канальной головки)
# Но intermediate = 256 (после B2). Здесь используем intermediate для word-aux,
# чтобы вынести word-смысл уже с середины энкодера.
word_head = WordAuxCTCHead(
    d_enc=model._d_mid,           # 256 — intermediate depth
    word_vocab_size=word_vocab.size,
    blank_id=word_vocab.BLANK_ID,
).to(device)

ctc_loss_fn = CTCLoss(blank_id=vocab.blank_id)
all_params = list(model.parameters()) + list(word_head.parameters())
optimizer = build_novograd(all_params, lr=HP["lr"], betas=(0.95, 0.5),
                            weight_decay=HP["weight_decay"])
total_steps = len(train_loader) * HP["max_epochs"]
scheduler = build_cosine_warmup(optimizer, total_steps=total_steps,
                                 warmup_steps=HP["warmup_steps"])

ckpt_dir = paths.ckpt_root / "01b_word_aux"
best_cer = float("inf")
history = []
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")
print(f"WordAuxHead params: {sum(p.numel() for p in word_head.parameters()):,}")

In [ ]:
for epoch in tqdm(range(HP["max_epochs"]), desc="epochs"):
    model.train(); word_head.train(); spec_aug.train()
    optimizer.zero_grad()
    grad_step = 0

    for step, batch in enumerate(tqdm(train_loader, desc=f"epoch {epoch}", leave=False)):
        audio = batch.audio.to(device)
        audio_lengths = batch.audio_lengths.to(device)
        targets = batch.targets.to(device)
        target_lengths = batch.target_lengths.to(device)

        # Вычислить word targets на лету из batch.transcriptions
        word_targets, word_tlens = _encode_words(batch.transcriptions, word_vocab)
        word_targets = word_targets.to(device)
        word_tlens = word_tlens.to(device)

        with torch.no_grad():
            mel = mel_extractor(audio)
        mel_lengths = (
            (audio_lengths + HP_FIXED["hop_length"] - 1) // HP_FIXED["hop_length"]
        ).clamp(max=mel.size(-1))
        mel = spec_aug(mel, mel_lengths)

        with torch.autocast(device_type=device.type, dtype=torch.float16,
                             enabled=(device.type == "cuda")):
            out = model(mel, mel_lengths)

        with torch.autocast(device_type=device.type, enabled=False):
            loss_main = ctc_loss_fn(
                out.log_probs.float(), targets, out.output_lengths, target_lengths
            )
            # --- Word-Aux CTC: точка внедрения ---
            if out.intermediate is not None:
                loss_word = word_head(
                    out.intermediate, out.output_lengths, word_targets, word_tlens
                )
            else:
                loss_word = torch.tensor(0.0, device=device)

        loss = loss_main + HP["word_aux_weight"] * loss_word

        (loss / HP["grad_accum"]).backward()
        grad_step += 1
        if grad_step % HP["grad_accum"] == 0:
            torch.nn.utils.clip_grad_norm_(all_params, max_norm=1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()

    # Validation
    model.eval(); word_head.eval(); spec_aug.eval()
    all_refs, all_hyps, all_spks = [], [], []

    with torch.no_grad():
        for batch in dev_loader:
            audio = batch.audio.to(device)
            audio_lengths = batch.audio_lengths.to(device)
            mel = mel_extractor(audio)
            mel_lengths = (
                (audio_lengths + HP_FIXED["hop_length"] - 1) // HP_FIXED["hop_length"]
            ).clamp(max=mel.size(-1))
            out = model(mel, mel_lengths)
            hyps = greedy_decode(out.log_probs, out.output_lengths, vocab)
            all_refs.extend(batch.transcriptions)
            all_hyps.extend(hyps)
            all_spks.extend(batch.spk_ids)

    ref_words = [digits_to_words(r) for r in all_refs]
    val_cer = compute_cer(ref_words, all_hyps)
    spk_cer = compute_per_speaker_cer(ref_words, all_hyps, all_spks)
    max_spk_cer = max(spk_cer.values()) if spk_cer else 0.0
    history.append({"epoch": epoch, "val_cer": val_cer, "max_spk_cer": max_spk_cer})
    tqdm.write(f"epoch {epoch:3d} | val_cer={val_cer:.4f} | max_spk_cer={max_spk_cer:.4f}")

    if val_cer < best_cer:
        best_cer = val_cer
        ckpt_path = save_best(
            model,
            meta=dict(arch="quartznet10x4", hparams=HP, val_cer=val_cer, epoch=epoch),
            ckpt_dir=ckpt_dir,
        )
        tqdm.write(f"  Checkpoint saved: {ckpt_path}")

## Итог

In [ ]:
import pandas as pd

df = pd.DataFrame(history)
print(f"Best val CER: {best_cer:.4f}")
print(f"Checkpoint dir: {ckpt_dir}")
print()
print(df.tail(10).to_string(index=False))